# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² colorectal cancer survivors dataset using the `mlcroissant` library. We will follow a clear, step-by-step approach to explore the data using entity `@id`s for record sets, fields, and columns as per the Croissant schema.

### Dataset Source

[FAIR² data package JSON-LD](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) ([doi:10.71728/senscience.qs2f-h81p](https://doi.org/10.71728/senscience.qs2f-h81p))

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"\033[1mDataset name:\033[0m {metadata.name}")
print(f"\033[1mDescription:\033[0m {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and their structure. We'll enumerate all available record sets, then review field `@id`s for each, using the Croissant metadata.

In [ ]:
# List record sets (by @id) and their fields
print("Available record sets and their fields:\n")
recordset_ids = []
for rs in metadata.record_sets:
    print(f"➡️ RecordSet @id: {rs.id} - name: {rs.name}")
    recordset_ids.append(rs.id)
    for field in rs.fields:
        print(f"    ⮑ Field @id: {field.id} - name: {field.name}")
    print()
if not recordset_ids:
    print('No record sets directly listed in the root metadata. The `mlcroissant` library may infer them from DataFiles.')

Most real-world Croissant datasets (including this one) describe tabular data through a primary record set, often with the patient/row-level data as the main record set. The actual `@id`s can be determined interactively or, if not provided at the root, by examining the loaded dataset further.

Let's get a list of available record sets via the `dataset.record_set_ids` interface and inspect their fields:

In [ ]:
# Get record set IDs directly available via the mlcroissant loader
print("Discovered record set IDs:")
for rs_id in dataset.record_set_ids:
    print(f"- {rs_id}")

# Let's choose the record set with clinical tabular data (assume first for now)
chosen_recordset_id = dataset.record_set_ids[0]  # Adjust if needed according to printed list
print(f"\nInspecting fields for record set: {chosen_recordset_id}")
fields = dataset.fields(chosen_recordset_id)
for field in fields:
    print(f"Field: {field['name']} - @id: {field['@id']}")

## 3. Data Extraction
Now, let's load the records from the chosen main record set into a pandas DataFrame for further analysis. We'll also demonstrate how to handle multiple record sets if present.

In [ ]:
# Load all record sets into DataFrames (use @id)
dataframes = {}
print("Loading all records for available record sets...\n")

for rs_id in dataset.record_set_ids:
    print(f"Loading: {rs_id}")
    rows = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(rows)
    dataframes[rs_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Number of rows: {len(df)}\n")

# Pick the main record set and show first rows
main_rs_id = chosen_recordset_id
main_df = dataframes[main_rs_id]
print(f"\033[1mColumns in record set {main_rs_id}:\033[0m")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
We will now process the data. Let's identify numeric fields by their `@id` and field names, then perform standard EDA:
- Filter based on a numeric field
- Normalize the distribution
- Group by a key categorical field

**All references to fields/columns use their `@id`.**

In [ ]:
# Identify numeric fields for EDA
numeric_fields = []
field_types = {}
for field in dataset.fields(main_rs_id):
    if field.get('dataType', '').lower() in (['float', 'integer', 'number']):
        numeric_fields.append(field['@id'])
    field_types[field['@id']] = field.get('dataType', '')

print("Numeric fields available (by @id):")
for nf in numeric_fields:
    print(f"- {nf}")

if not numeric_fields:
    print("WARNING: No numeric fields auto-identified; please review main_df.dtypes or schema.")

# Pick a field likely to be numeric (fallback: try to infer from DataFrame)
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    # Try to pick a column that is numeric (based on DataFrame types)
    possible_numeric = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
    if possible_numeric:
        numeric_field_id = possible_numeric[0]
    else:
        raise Exception('No numeric field found for analysis.')

print(f"\nChosen numeric field for EDA: {numeric_field_id}\n")

# Filtering and normalization
threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].dtype.kind in 'fi' else 10
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (first 5 rows):\n")
print(filtered_df.head())

# Normalize
norm_col = numeric_field_id + '_normalized'
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:\n")
print(filtered_df[[numeric_field_id, norm_col]].head())

# Choose a categorical field to group by
cat_fields = [field['@id'] for field in dataset.fields(main_rs_id) if field.get('dataType', '').lower() == 'text']
if not cat_fields:
    # Fallback: find object/string type columns
    cat_fields = [col for col in main_df.columns if main_df[col].dtype == object]
if cat_fields:
    group_field_id = cat_fields[0]
    print(f"\nGrouping by field: {group_field_id}\n")
    # Only group if the selected column exists in the filtered_df
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id} (first 5 groups):")
        print(grouped_df.head())
    else:
        print(f"{group_field_id} not in dataframe; skipping groupby.")
else:
    print("No categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the chosen numeric field after normalization, and optionally show a bar plot for group means if grouping succeeded.

We'll use matplotlib and seaborn for basic plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for normalized values
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[norm_col], kde=True)
plt.title(f"Distribution of normalized field: {numeric_field_id}")
plt.xlabel(norm_col)
plt.ylabel("Count")
plt.show()

# If grouped_df exists, plot barplot
if 'grouped_df' in locals():
    plt.figure(figsize=(10,5))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We loaded clinicopathological data from the FAIR² colorectal cancer survivors dataset using the Croissant schema and the mlcroissant library (referencing record sets and fields by their `@id`).
- Key record sets, fields, and data types were identified programmatically by `@id`.
- Demonstrated filtering and normalization on a numeric field, and grouping by a categorical attribute.
- Basic visualizations revealed the distribution and central tendencies in the selected features.

**You can adapt this notebook to perform more detailed exploration and modeling on different record sets or fields of interest, always referencing entities by their schema `@id`.**